<a href="https://colab.research.google.com/github/gowtham-garimella/AI-RESUME-PARSER/blob/main/AI_RESUME_PARSER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================
# AI Resume Screening Web App (No Login - Clean Version)
# ================================

!pip -q install gradio pdfplumber python-docx sentence-transformers scikit-learn

import gradio as gr
import pdfplumber, docx, re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Load AI model
model = SentenceTransformer('all-MiniLM-L6-v2')

# -----------------------------
# TEXT EXTRACTION
# -----------------------------
def extract_text(file):
    if file.name.endswith(".pdf"):
        text = ""
        with pdfplumber.open(file.name) as pdf:
            for p in pdf.pages:
                text += p.extract_text() or ""
        return text

    elif file.name.endswith(".docx"):
        doc = docx.Document(file.name)
        return "\n".join([p.text for p in doc.paragraphs])

    elif file.name.endswith(".txt"):
        return open(file.name).read()

    return ""

# -----------------------------
# CLEAN TEXT
# -----------------------------
def clean(text):
    return re.sub(r"[^a-zA-Z0-9 ]", " ", text.lower())

# -----------------------------
# KEYWORD GAP DETECTION
# -----------------------------
def keyword_gap(resume, jd):
    stopwords = set(ENGLISH_STOP_WORDS)

    resume_words = set([w for w in resume.split() if w not in stopwords and len(w) > 2])
    jd_words = set([w for w in jd.split() if w not in stopwords and len(w) > 2])

    return list(jd_words - resume_words)[:10]

# -----------------------------
# MAIN ANALYSIS FUNCTION
# -----------------------------
def analyze(jd, files):

    if not jd:
        return "⚠️ Please enter job description"

    if not files:
        return "⚠️ Please upload at least one resume"

    jd_clean = clean(jd)
    results = []

    for file in files:
        resume = extract_text(file)
        res_clean = clean(resume)

        emb1 = model.encode([res_clean])
        emb2 = model.encode([jd_clean])

        score = cosine_similarity(emb1, emb2)[0][0] * 100
        missing = keyword_gap(res_clean, jd_clean)

        results.append((file.name, round(score,2), missing, res_clean))

    # Sort by score
    results.sort(key=lambda x: x[1], reverse=True)

    # Build output
    output = "📊 RESUME ANALYSIS DASHBOARD\n\n"

    for name, score, missing, text in results:
        output += f"📄 {name}\n"
        output += f"Score: {score}/100\n"

        if score >= 80:
            output += "🔥 Strong match\n"
        elif score >= 60:
            output += "👍 Moderate match\n"
        else:
            output += "⚠️ Low match\n"

        if missing:
            output += "🔑 Missing Skills: " + ", ".join(missing) + "\n"

        output += "💡 Suggestions:\n"

        if "project" not in text:
            output += "- Add strong project descriptions\n"
        if "experience" not in text:
            output += "- Add experience section\n"
        if "skills" not in text:
            output += "- Add skills section\n"

        output += "- Tailor resume to job description\n"
        output += "--------------------------\n"

    return output

# -----------------------------
# UI (GRADIO)
# -----------------------------
with gr.Blocks(css="""
body {background: linear-gradient(135deg, #1e3c72, #2a5298); color: white;}
""") as app:

    gr.Markdown("# 🤖 AI Resume Screening System")
    gr.Markdown("### Upload Resumes + Paste JD → Get AI Score")

    jd = gr.Textbox(label="📋 Job Description", lines=8)
    files = gr.File(label="📄 Upload Resumes", file_count="multiple")

    analyze_btn = gr.Button("🔍 Analyze")

    output = gr.Textbox(label="📊 Output", lines=20)

    analyze_btn.click(analyze, inputs=[jd, files], outputs=output)

app.launch(share=True)